Notebook 03: Model 1 Training (Property Extraction)
 ============================================================
 Two-stage training of the PropertyYOLO model:
 Stage 1: Detection training (YOLOv8 backbone, COCO pretrained)
 Stage 2: Property regression training (backbone frozen)

 ⚠ Requires GPU! Run in Google Colab with GPU runtime.
 ============================================================


## Stage 1: YOLOv8 Backbone Training
**Objective:** Train the base YOLOv8 model to detect and classify objects (bounding boxes and class labels). 
**Note:** To bypass Google Drive I/O throttling, we copy the dataset to the local Colab SSD before training.

=== Cell 1: Python (Environment Setup & SSD Transfer) ===

In [ ]:
import os
import yaml
from google.colab import drive

# Mount Drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

# Copy dataset to high-speed Local SSD
print("Copying dataset to Local SSD...")
!mkdir -p /content/dataset
!cp -r /content/drive/MyDrive/xray_detection/data/raw /content/dataset/

# Build local data.yaml
yaml_data = {
    'path': '/content/dataset/raw',
    'train': ['sixray/train/images', 'opixray/train/images'],
    'val': ['sixray/valid/images', 'opixray/test/images'],
    'nc': 5,
    'names': ['gun', 'knife', 'wrench', 'pliers', 'scissors']
}
with open("/content/data.yaml", 'w') as file:
    yaml.dump(yaml_data, file, default_flow_style=False)
print("Environment Ready!")

=== Cell 2: Python (Training Execution) ===

In [ ]:
from ultralytics import YOLO
import wandb

# Initialize Weights & Biases
wandb.login()

# Start Stage 1 Training
model = YOLO('yolov8m.pt')
results = model.train(
    data="/content/data.yaml",
    epochs=100, 
    batch=16, 
    imgsz=640,
    project="/content/drive/MyDrive/xray_detection/results",
    name="stage1_detection",
    device=0 
)

## Stage 2: Property Head Training
**Objective:** Freeze the backbone and train the custom 11-dimensional Property Regression Head using the Stage 1 weights.

In [ ]:
import torch
from torch.utils.data import DataLoader
from src.model1.architecture import PropertyYOLO
from src.model1.loss import MultiTaskLoss
from src.model1.train import Trainer
from src.dataset.xray_dataset import XRayDataset, collate_fn

# 1. Instantiate the PyTorch Dataset/DataLoader
train_dataset = XRayDataset(
    image_dir='/content/dataset/raw',
    label_dir='/content/dataset/raw',
    property_csv='/content/dataset/raw/properties.csv',
    image_size=640
)
train_loader = DataLoader(
    train_dataset, 
    batch_size=16, 
    shuffle=True, 
    num_workers=2, 
    collate_fn=collate_fn
)

# 2. Initialize the model with Stage 1 weights
model = PropertyYOLO(
    model_size='yolov8m',
    num_classes=6, # 5 threats + 1 background
    num_properties=11,
    weights_path='weights/yolov8_xray_stage1_final.pt'
)

# 3. Freeze backbone
model.set_training_stage(2)

# 4. Trigger the 50-epoch training run
config = {'model1': {'stage2': {'epochs': 50, 'learning_rate': 0.001}}}
trainer = Trainer(
    config=config,
    model=model,
    device='cuda' if torch.cuda.is_available() else 'cpu'
)
trainer.train_stage2(train_loader, epochs=50)